In [1]:
import ee

In [2]:
ee.Authenticate()

True

In [3]:
# Initialize Google Earth Engine project
ee.Initialize(project='ee-jonstar')

## CONUS Geometry

In [4]:
"""
Export coordinates for each urban area
KEY: [utm zone, boolean T/F for northern/southern hemisphere, utm x, utm y]
"""
export_coords = {
    'KBWI':[18, True, 292000, 4372200],
    'KJFK':[18, True, 561166, 4546228],
    'KPHX':[12, True, 356592, 3758141],
    'KMIA':[17, True, 503991, 2915332],
    'KORD':[16, True, 386195, 4680677],
    'KDEN':[13, True, 466349, 4442816],
    'KSEA':[10, True, 518589, 5311884],
    'KSFO':[10, True, 541523, 4206919],
    'KLAX':[11, True, 346524, 3803779],
    'KATL':[16, True, 703143, 3787639],
    'CYYZ':[17, True, 566749, 4867878],
    'MMMX':[14, True, 426632, 2177328],
    'KLAS':[11, True, 618776, 4038562],
    'KSLC':[12, True, 371135, 4538777],
    'KDFW':[14, True, 639754, 3678675],
    'KIAH':[15, True, 232521, 3334892],
    'KMSY':[15, True, 723933, 3361201],
    'KSTL':[15, True, 672050, 4323318],
    'KMSP':[15, True, 433510, 5031071],
    'KJAX':[17, True, 401561, 3376745],
    'KCLT':[17, True, 472912, 3938294],
    'KPHL':[18, True, 445926, 4469797],
    'KSAN':[11, True, 468102, 3666860],
    'TJSJ':[19, True, 732184, 2071833],
    'CYUL':[18, True, 564088, 5089380],
    'MMGT':[13, True, 621576, 2326245],
    'MMMY':[14, True, 318956, 2885247],
    'MMUN':[16, True, 470776, 2369100],
    'KBIL':[12, True, 655301, 5109344],
    'MGGT':[15, True, 711929, 1659823],
    'MROC':[16, True, 772982, 1148166],
    'MUHA':[17, True, 310663, 2566532],
    'MDSD':[19, True, 354586, 2108039],
    'MHTG':[16, True, 405549, 1616590],
    'MNMG':[16, True, 524984, 1388932],
    'MPTO':[17, True, 610881, 1043658],
    'SKBO':[18, True, 546442, 561743],
    'SPJC':[18, False, 262348, 8709133],
    'SEQM':[17, True, 733395, 16923],
    'SCEL':[19, False, 298834, 6329569],
    'SAEZ':[21, False, 326030, 6204959],
    'SBGR':[23, False, 294104, 7446208],
    'SBEG':[20, False, 762422, 9716468],
    'SCCI':[19, False, 319805, 4162998],
    'SLLP':[19, False, 548109, 8213756],
    'SUMU':[21, False, 545458, 6223320],
    'SBBR':[22, False, 783269, 8287283],
    'SVMI':[19, True, 687497, 1176029]
}

cities = list(export_coords.keys())

In [5]:
# Pull points to make data for a specific city (look above for options)
city = cities[8]
print(city)
city_export = export_coords[city]
if city_export[1]:
    crs_prefix = '326'
else:
    crs_prefix = '327'
utm_proj = f'EPSG:{crs_prefix}{city_export[0]}'
point = ee.Geometry.Point([city_export[2]+30*2999/2,city_export[3]-30*2999/2], utm_proj)
export_region = point.buffer(distance=44990, proj=utm_proj).bounds(proj=utm_proj)
city_export

KLAX


[11, True, 346524, 3803779]

## Landsat Initialization

In [6]:
# Initialize LCZ ImageCollection
lcz = ee.ImageCollection("RUB/RUBCLIM/LCZ/global_lcz_map/latest")
lcz_map = lcz.select('LCZ_Filter').first()

## Export stuff

In [7]:
# Export LCZ image
for city in cities:
    city_export = export_coords[city]
    if city_export[1]:
        crs_prefix = '326'
    else:
        crs_prefix = '327'
    utm_proj = f'EPSG:{crs_prefix}{city_export[0]}'
    point = ee.Geometry.Point([city_export[2]+30*2999/2,city_export[3]-30*2999/2], utm_proj)
    export_region = point.buffer(distance=44990, proj=utm_proj).bounds(proj=utm_proj)
    
    task = ee.batch.Export.image.toDrive(
        lcz_map, description=f'LCZ_{city}', fileFormat='GeoTIFF',
        region=export_region, crs=utm_proj, scale=30, formatOptions={'cloudOptimized':True})
    task.start()